# County Rainfall and Planting Advisory

**Analyst:** Exaultant  
**Libraries:** NumPy (plus the standard-library `csv` module)

A farmer-support NGO sends SMS planting advice to farmers in five counties. This notebook cleans the hand-typed Machakos station data, analyses monthly rainfall with NumPy, writes `rainfall_summary.csv` and ends with a planting advisory.

In [1]:
import csv
import numpy as np

counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]

clean_rows = [

    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],   # Nairobi

    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],    # Nakuru

    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],   # Kisumu

    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],   # Mombasa

]

machakos_raw = ["48", "35", "88", "N/A", "140", " 20 ",

                "9", "12", "", "45", "182", "96"]

months = np.array(["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                   "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])

## Task 1: Clean the Machakos data

**Approach:** `clean_rainfall()` tries `int(value)` on each reading. `int()` already ignores spaces (so `" 20 "` becomes 20) and raises a `ValueError` for `"N/A"` and `""`, which I catch with a specific `except ValueError` and mark as missing. Missing months are then replaced with the rounded average of the valid readings, and the imputed months are returned so they can be printed.

In [2]:
def clean_rainfall(raw_values):
    """Convert raw rainfall strings to integers and impute invalid entries.

    Each value is converted with int(); a ValueError (e.g. "N/A" or an empty
    string) marks the month as invalid. Invalid months are replaced with the
    rounded average of the valid readings.

    Returns (cleaned_list, imputed_indices, fill_value).
    """
    cleaned = []
    invalid_idx = []
    for i, value in enumerate(raw_values):
        try:
            cleaned.append(int(value))
        except ValueError:
            cleaned.append(None)
            invalid_idx.append(i)

    valid = [v for v in cleaned if v is not None]
    fill_value = round(sum(valid) / len(valid))
    cleaned = [fill_value if v is None else v for v in cleaned]
    return cleaned, invalid_idx, fill_value


machakos_clean, imputed_idx, fill_value = clean_rainfall(machakos_raw)

print("Cleaned Machakos row:", machakos_clean)
print("Imputed months     :", [str(months[i]) for i in imputed_idx])
print("Imputed with       :", fill_value, "mm (rounded average of the 10 valid months)")

Cleaned Machakos row: [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]
Imputed months     : ['Apr', 'Sep']
Imputed with       : 68 mm (rounded average of the 10 valid months)


## Task 2: Combine into one 5 × 12 array

**Approach:** Stack the four clean rows and the cleaned Machakos row into one integer array with `np.array()`, then confirm `.shape` and `.dtype`.

In [3]:
rain = np.array(clean_rows + [machakos_clean])

print("Shape:", rain.shape)
print("dtype:", rain.dtype)
print(rain)

Shape: (5, 12)
dtype: int64
[[ 64  56  92 219 176  35  18  24  31  61 165 104]
 [ 35  42  70 145 120  78  96 105  72  55  60  38]
 [ 58  90 150 240 180  95  70  85  90  88 130  95]
 [ 30  15  55 150 240 110  90  65  60  90 100  70]
 [ 48  35  88  68 140  20   9  12  68  45 182  96]]


## Task 3: Annual totals and monthly averages

**Approach:** `sum(axis=1)` adds across the months of each county (annual totals), and `mean(axis=0)` averages down the counties for each month.

In [4]:
annual_totals = rain.sum(axis=1)
monthly_avg = rain.mean(axis=0)

print("Annual total rainfall (mm):")
for county, total in zip(counties, annual_totals):
    print(f"  {county:<9}{total:>6,}")

print("\nAverage rainfall per month across all counties (mm):")
for month, avg in zip(months, monthly_avg):
    print(f"  {month}: {avg:6.1f}")

Annual total rainfall (mm):
  Nairobi   1,045
  Nakuru      916
  Kisumu    1,371
  Mombasa   1,075
  Machakos    811

Average rainfall per month across all counties (mm):
  Jan:   47.0
  Feb:   47.6
  Mar:   91.0
  Apr:  164.4
  May:  171.2
  Jun:   67.6
  Jul:   56.6
  Aug:   58.2
  Sep:   64.2
  Oct:   67.8
  Nov:  127.4
  Dec:   80.6


## Task 4: Wettest month per county

**Approach:** `np.argmax(rain, axis=1)` returns the column index of each row's maximum; indexing the `months` array with it gives month names, and `zip()` pairs them with the counties.

In [5]:
wettest_idx = np.argmax(rain, axis=1)
wettest_month = months[wettest_idx]
wettest_mm = rain.max(axis=1)

for county, month, mm in zip(counties, wettest_month, wettest_mm):
    print(f"{county:<9} wettest month: {month} ({mm} mm)")

Nairobi   wettest month: Apr (219 mm)
Nakuru    wettest month: Apr (145 mm)
Kisumu    wettest month: Apr (240 mm)
Mombasa   wettest month: May (240 mm)
Machakos  wettest month: Nov (182 mm)


## Task 5: Dry months (below 50 mm)

**Approach:** The boolean mask `rain < 50` marks dry months; summing it along `axis=1` counts the `True` values in each row.

In [6]:
dry_mask = rain < 50
dry_months = dry_mask.sum(axis=1)

for county, n in zip(counties, dry_months):
    print(f"{county:<9} dry months: {n}")

Nairobi   dry months: 4
Nakuru    dry months: 3
Kisumu    dry months: 0
Mombasa   dry months: 2
Machakos  dry months: 6


## Task 6: Planting window (best two consecutive months)

**Approach:** The shifted-array trick `rain[:, :-1] + rain[:, 1:]` gives the combined rainfall of every pair of consecutive months (Jan+Feb, Feb+Mar, ...). `argmax(axis=1)` finds the best pair for each county, and the start month and the following month give the window.

In [7]:
pair_sums = rain[:, :-1] + rain[:, 1:]          # shape (5, 11)
best_start = np.argmax(pair_sums, axis=1)
best_total = pair_sums.max(axis=1)
planting_window = np.char.add(np.char.add(months[best_start], "-"), months[best_start + 1])

for county, window, mm in zip(counties, planting_window, best_total):
    print(f"{county:<9} planting window: {window} ({mm} mm combined)")

Nairobi   planting window: Apr-May (395 mm combined)
Nakuru    planting window: Apr-May (265 mm combined)
Kisumu    planting window: Apr-May (420 mm combined)
Mombasa   planting window: Apr-May (390 mm combined)
Machakos  planting window: Nov-Dec (278 mm combined)


## Task 7: Write `rainfall_summary.csv`

**Approach:** Use the `csv` module's `DictWriter` to write one row per county with the five required columns.

In [8]:
with open("rainfall_summary.csv", "w", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["county", "annual_total_mm", "wettest_month", "dry_months", "planting_window"],
    )
    writer.writeheader()
    for county, total, month, dry, window in zip(
        counties, annual_totals, wettest_month, dry_months, planting_window
    ):
        writer.writerow({
            "county": county,
            "annual_total_mm": int(total),
            "wettest_month": str(month),
            "dry_months": int(dry),
            "planting_window": str(window),
        })

with open("rainfall_summary.csv") as f:
    print(f.read())

county,annual_total_mm,wettest_month,dry_months,planting_window
Nairobi,1045,Apr,4,Apr-May
Nakuru,916,Apr,3,Apr-May
Kisumu,1371,Apr,0,Apr-May
Mombasa,1075,May,2,Apr-May
Machakos,811,Nov,6,Nov-Dec



## Planting Advisory

Farmers in Nairobi, Nakuru, Kisumu and Mombasa should plan to plant in April-May, which is the wettest two-month window in all four counties (from 265 mm combined in Nakuru to 420 mm in Kisumu). Kisumu is the most reliable county, with 1,371 mm a year and no dry months, whereas Nairobi has 4 dry months (June to September), so Nairobi crops sown in April-May should be ones that mature before the dry spell. Machakos is the driest county (811 mm a year and 6 dry months) and its rains come later, so farmers there should plant in November-December (278 mm combined) and choose drought-tolerant crops. Mombasa's wettest month is May (240 mm), so seed and inputs should be ready by the end of March. One caveat: two Machakos readings (April and September) were missing and filled with the 68 mm average, so Machakos's annual total and dry-month count may be slightly off and should be confirmed with the station before advice is sent.

## Data Quality Notes

- The Machakos row was typed by hand: `"N/A"` (April) and an empty string (September) could not be converted, and `" 20 "` had stray spaces (`int()` handled it correctly as 20).
- April and September were imputed with the rounded mean of the 10 valid months (68 mm). April is normally a wet month in the other counties, so 68 mm may understate Machakos's real rainfall and bias its annual total and planting window.
- Only one year of data is available, so the windows reflect a single season and should not be treated as long-term averages.
- Values were assumed to be in mm with no further validation (for example no range checks), since no other errors were found.